# QDSV/QIntent BP-Soft qLDPC-Style Decoder Reranking

This notebook is the next step after the random sparse benchmark. It connects QDSV/QIntent to a real decoder-style signal source: a lightweight belief-propagation (BP) soft decoder over a sparse parity-check matrix.

The experiment is still a toy qLDPC-style benchmark, not a production qLDPC code or a hardware decoder. The purpose is narrower:

- generate sparse syndrome constraints;
- sample noisy error patterns;
- run BP message passing to obtain soft posterior evidence;
- enumerate syndrome-compatible correction candidates;
- compare a BP-confidence baseline against QDSV structured semantic reranking;
- preserve an auditable trace without exposing the private QDSV scoring formula.

This tests whether richer decoder evidence helps QDSV avoid the ambiguity seen in the previous random sparse benchmark.


## 1. Install QIntent


In [ ]:
!pip install -q --upgrade "git+https://github.com/qdsvquantum-afk/qintent.git"


In [ ]:
import json
import math
import random
from itertools import combinations
from pathlib import Path

import pandas as pd

from qintent import QIntentClient

API_URL = "https://qdsv-api-568788785178.us-central1.run.app/api"
client = QIntentClient(api_url=API_URL, timeout=60)

client.spec()["public_limits"]


## 2. Benchmark configuration

`N_SAMPLES` controls public API usage. Each sample performs one QIntent `run()` call. The BP decoder itself runs locally in the notebook.


In [ ]:
SEED = 260717
N_QUBITS = 24
M_CHECKS = 8
N_SAMPLES = 40
MAX_CANDIDATE_WEIGHT = 3
TRUE_ERROR_MAX_WEIGHT = 2
COLUMN_WEIGHT = 3
PHYSICAL_ERROR_RATE = 0.14

rng = random.Random(SEED)


## 3. Generate sparse qLDPC-style check structure


In [ ]:
check_columns = {}
used_columns = set()

for q in range(N_QUBITS):
    while True:
        weight = COLUMN_WEIGHT if rng.random() < 0.75 else 2
        positions = rng.sample(range(M_CHECKS), weight)
        column = [0] * M_CHECKS
        for pos in positions:
            column[pos] = 1
        column = tuple(column)
        if column not in used_columns and any(column):
            used_columns.add(column)
            check_columns[q] = column
            break

H = [[check_columns[q][check] for q in range(N_QUBITS)] for check in range(M_CHECKS)]
logical_sensitive_qubits = set(rng.sample(range(N_QUBITS), max(5, N_QUBITS // 3)))

pd.DataFrame([
    {
        "qubit": q,
        "check_column": "".join(str(bit) for bit in column),
        "logical_sensitive": q in logical_sensitive_qubits,
    }
    for q, column in check_columns.items()
])


## 4. BP decoder and candidate generation

The BP decoder is a lightweight sum-product message-passing decoder over the binary parity-check matrix. It produces posterior error probabilities for each qubit. Those soft outputs become prepared evidence for QDSV/QIntent.

Ground-truth labels are used only for evaluation; they are not used as QIntent scoring signals.


In [ ]:
def xor_syndrome(qubits):
    return tuple(
        sum((1 if q in qubits else 0) * H[check][q] for q in range(N_QUBITS)) % 2
        for check in range(M_CHECKS)
    )


def residual_error(true_error, correction):
    return frozenset(set(true_error) ^ set(correction))


def logical_failure_proxy(residual):
    # Toy proxy: a small residual touching a logical-sensitive qubit is treated as risky.
    return bool(residual & logical_sensitive_qubits) and len(residual) <= 3


def channel_priors(true_error):
    priors = []
    for q in range(N_QUBITS):
        if q in true_error:
            value = rng.uniform(0.08, 0.38)
        else:
            value = rng.uniform(0.03, 0.28)
        if q in logical_sensitive_qubits and rng.random() < 0.22:
            value = rng.uniform(0.18, 0.48)
        if q not in true_error and rng.random() < 0.08:
            value = rng.uniform(0.20, 0.45)
        priors.append(value)
    return priors


def bp_soft_decode(syndrome, priors, max_iter=20):
    eps = 1e-12
    checks_for_var = [[check for check in range(M_CHECKS) if H[check][q]] for q in range(N_QUBITS)]
    vars_for_check = [[q for q in range(N_QUBITS) if H[check][q]] for check in range(M_CHECKS)]
    prior_llr = [math.log((1 - priors[q] + eps) / (priors[q] + eps)) for q in range(N_QUBITS)]

    q_msg = {(check, q): prior_llr[q] for q in range(N_QUBITS) for check in checks_for_var[q]}
    r_msg = {(check, q): 0.0 for q in range(N_QUBITS) for check in checks_for_var[q]}
    hard = [0] * N_QUBITS

    for iteration in range(max_iter):
        for check in range(M_CHECKS):
            variables = vars_for_check[check]
            for q in variables:
                product = 1.0
                for other_q in variables:
                    if other_q == q:
                        continue
                    product *= math.tanh(max(-30, min(30, q_msg[(check, other_q)])) / 2)
                product = max(-1 + eps, min(1 - eps, ((-1) ** syndrome[check]) * product))
                r_msg[(check, q)] = 2 * math.atanh(product)

        for q in range(N_QUBITS):
            connected_checks = checks_for_var[q]
            total = prior_llr[q] + sum(r_msg[(check, q)] for check in connected_checks)
            hard[q] = 1 if total < 0 else 0
            for check in connected_checks:
                q_msg[(check, q)] = prior_llr[q] + sum(
                    r_msg[(other_check, q)]
                    for other_check in connected_checks
                    if other_check != check
                )

    posterior_llr = [
        prior_llr[q] + sum(r_msg[(check, q)] for check in checks_for_var[q])
        for q in range(N_QUBITS)
    ]
    posterior_error_prob = [
        1 / (1 + math.exp(max(-30, min(30, llr))))
        for llr in posterior_llr
    ]
    return posterior_error_prob, hard, iteration + 1


def enumerate_candidates(syndrome, true_error, posterior_error_prob, scenario_id):
    raw = []
    for weight in range(1, MAX_CANDIDATE_WEIGHT + 1):
        for qubits in combinations(range(N_QUBITS), weight):
            if xor_syndrome(qubits) != syndrome:
                continue
            log_probability = 0.0
            for q in range(N_QUBITS):
                p_error = max(1e-9, min(1 - 1e-9, posterior_error_prob[q]))
                log_probability += math.log(p_error if q in qubits else 1 - p_error)
            overlap = sum(q in logical_sensitive_qubits for q in qubits)
            residual = residual_error(true_error, qubits)
            raw.append((
                qubits,
                weight,
                overlap,
                log_probability,
                len(residual) == 0,
                logical_failure_proxy(residual),
            ))

    if len(raw) < 2:
        return []

    log_values = [item[3] for item in raw]
    low, high = min(log_values), max(log_values)
    sorted_logs = sorted(log_values, reverse=True)
    decoder_margin = 0 if len(sorted_logs) < 2 else int(round(min(1000, max(0, (sorted_logs[0] - sorted_logs[1]) * 180))))

    rows = []
    for candidate_index, (qubits, weight, overlap, log_probability, exact, logical_fail) in enumerate(raw):
        decoder_confidence = 1000 if high == low else int(round(450 + 550 * (log_probability - low) / (high - low)))
        rows.append({
            "scenario_id": scenario_id,
            "candidate_index": candidate_index,
            "correction_qubits": " ".join(str(q) for q in qubits),
            "candidate_weight": weight,
            "observed_syndrome": "".join(str(bit) for bit in syndrome),
            "predicted_syndrome": "".join(str(bit) for bit in syndrome),
            "syndrome_support": max(0, 1000 - 20 * max(0, weight - 1)),
            "check_consistency": 1000,
            "logical_preservation": max(0, 950 - 160 * overlap - 15 * max(0, weight - 1)),
            "distance_safety": max(0, 930 - 140 * overlap - 15 * weight),
            "decoder_confidence": decoder_confidence,
            "decoder_margin": decoder_margin,
            "propagation_safety": max(0, 930 - 45 * weight - 45 * overlap),
            "syndrome_risk": min(1000, 30 + 10 * weight),
            "logical_risk": min(1000, 25 + 150 * overlap + 16 * weight),
            "syndrome_entropy_adjustment": 12 if weight == 2 and overlap == 0 else (-12 if overlap else 0),
            "exact_correction": exact,
            "logical_failure_proxy": logical_fail,
        })
    return rows


## 5. QIntent BP-soft reranking policy


In [ ]:
source = """
find_rows("candidate_index")
  .using_structured_semantic_score([
      block("syndrome", [
          signal("syndrome_support", influence=25, priority=2),
          signal("check_consistency", influence=20, priority=1),
          signal("decoder_margin", influence=20, priority=2),
      ], influence=28, priority=2, risk_adjustment="syndrome_risk", adjustments=[
          adjustment("syndrome_entropy_adjustment", influence=4),
      ]),
      block("logical_safety", [
          signal("logical_preservation", influence=32, priority=2),
          signal("distance_safety", influence=20, priority=2),
      ], influence=34, priority=2, risk_adjustment="logical_risk"),
      block("decoder", [
          signal("decoder_confidence", influence=60, priority=4),
          signal("propagation_safety", influence=15, priority=2),
      ], influence=45, priority=4),
  ], global_risk=0, profile="qldpc_bp_soft_decoder_rerank")
  .accept_if(threshold=600)
  .rank()
  .top_k(1)
"""

print(source)
# The run results below include the auditable decision trace.


## 6. Run BP-soft benchmark


In [ ]:
summary_rows = []
scenario_rows = []
attempts = 0
scenario_id = 0

while scenario_id < N_SAMPLES and attempts < 5000:
    attempts += 1
    true_weight = 1 if rng.random() < 0.45 else 2
    true_error = tuple(sorted(rng.sample(range(N_QUBITS), true_weight)))
    syndrome = xor_syndrome(true_error)
    priors = channel_priors(true_error)
    posterior_error_prob, hard_decision, bp_iterations = bp_soft_decode(syndrome, priors)
    rows = enumerate_candidates(syndrome, true_error, posterior_error_prob, scenario_id)
    if len(rows) < 2:
        continue

    candidates = pd.DataFrame(rows)
    baseline = candidates.sort_values(["decoder_confidence", "candidate_weight"], ascending=[False, True]).iloc[0]
    result = client.run(source, rows=rows)
    selected = result["result"]["selected_rows"][0]

    summary_rows.append({
        "scenario_id": scenario_id,
        "true_error": " ".join(str(q) for q in true_error),
        "true_weight": true_weight,
        "bp_iterations": bp_iterations,
        "candidate_count": len(rows),
        "baseline_qubits": baseline["correction_qubits"],
        "baseline_confidence": int(baseline["decoder_confidence"]),
        "baseline_logical_risk": int(baseline["logical_risk"]),
        "baseline_exact": bool(baseline["exact_correction"]),
        "baseline_failure": bool(baseline["logical_failure_proxy"]),
        "qdsv_qubits": selected["correction_qubits"],
        "qdsv_confidence": int(selected["decoder_confidence"]),
        "qdsv_logical_risk": int(selected["logical_risk"]),
        "qdsv_exact": bool(selected["exact_correction"]),
        "qdsv_failure": bool(selected["logical_failure_proxy"]),
        "risk_delta": int(baseline["logical_risk"]) - int(selected["logical_risk"]),
        "exact_delta": int(bool(selected["exact_correction"])) - int(bool(baseline["exact_correction"])),
        "failure_delta": int(bool(baseline["logical_failure_proxy"])) - int(bool(selected["logical_failure_proxy"])),
    })
    scenario_rows.append(rows)
    scenario_id += 1

summary = pd.DataFrame(summary_rows)
summary


## 7. Metrics


In [ ]:
metrics = {
    "samples": int(len(summary)),
    "attempts": int(attempts),
    "baseline_exact_rate": float(summary["baseline_exact"].mean()),
    "qdsv_exact_rate": float(summary["qdsv_exact"].mean()),
    "baseline_failure_proxy_rate": float(summary["baseline_failure"].mean()),
    "qdsv_failure_proxy_rate": float(summary["qdsv_failure"].mean()),
    "baseline_avg_logical_risk": float(summary["baseline_logical_risk"].mean()),
    "qdsv_avg_logical_risk": float(summary["qdsv_logical_risk"].mean()),
    "avg_risk_delta": float(summary["risk_delta"].mean()),
    "improved_risk_count": int((summary["risk_delta"] > 0).sum()),
    "worse_risk_count": int((summary["risk_delta"] < 0).sum()),
    "avg_exact_delta": float(summary["exact_delta"].mean()),
    "avg_failure_delta": float(summary["failure_delta"].mean()),
}

metrics


## 8. Inspect changed selections


In [ ]:
summary.sort_values(["risk_delta", "exact_delta", "failure_delta"], ascending=[False, False, False]).head(12)


In [ ]:
summary[summary["baseline_qubits"] != summary["qdsv_qubits"]].sort_values(
    ["risk_delta", "exact_delta", "failure_delta"],
    ascending=[False, False, False],
)


## 9. Save evidence


In [ ]:
evidence = {
    "api_url": API_URL,
    "profile": "qldpc_bp_soft_decoder_rerank",
    "benchmark_design": {
        "seed": SEED,
        "n_qubits": N_QUBITS,
        "m_checks": M_CHECKS,
        "n_samples": N_SAMPLES,
        "max_candidate_weight": MAX_CANDIDATE_WEIGHT,
        "true_error_max_weight": TRUE_ERROR_MAX_WEIGHT,
        "physical_error_rate": PHYSICAL_ERROR_RATE,
        "column_weight": COLUMN_WEIGHT,
        "construction": "sparse parity-check matrix + local BP soft decoder + low-weight syndrome-compatible candidate enumeration",
    },
    "check_columns": {str(k): list(v) for k, v in check_columns.items()},
    "logical_sensitive_qubits": sorted(logical_sensitive_qubits),
    "qintent_source": source,
    "summary": summary.to_dict(orient="records"),
    "metrics": metrics,
    "internal_formula_exposed": False,
}

Path("qdsv_qldpc_bp_soft_decoder_evidence.json").write_text(json.dumps(evidence, indent=2), encoding="utf-8")
summary.to_csv("qdsv_qldpc_bp_soft_decoder_summary.csv", index=False)

print("Saved:")
print("- qdsv_qldpc_bp_soft_decoder_evidence.json")
print("- qdsv_qldpc_bp_soft_decoder_summary.csv")


## Interpretation

This benchmark adds real decoder-style soft evidence. In the previous random sparse benchmark, some scenarios were observationally ambiguous under the available candidate features. Here, BP posterior probabilities and decoder margin provide richer evidence before QDSV/QIntent reranking.

The result should be interpreted carefully:

- QDSV/QIntent is not replacing BP.
- BP supplies soft candidate evidence.
- QDSV/QIntent acts as a structured post-decoding decision layer over that evidence.
- The private scoring formula remains hidden; only block-level public evidence and trace fields are exposed.
- This is still a toy sparse-check benchmark and not a production qLDPC decoder comparison.
